In [1]:
import pandas as pd
import json
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler,StandardScaler

current_dir = Path.cwd()
project_root = current_dir.parents[2]



with open(project_root/"SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/Domain_data.json", "r") as archivo:
    full_domain = json.load(archivo)

with open(project_root/"SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS_Domain_data.json", "r") as archivo:
    updrs_domain = json.load(archivo)

X_multiples= { 'X_STATS':project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS.csv', 
                    'X_V06+STATS': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+STATS.csv',
                      'X_V06+DELTA': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+Deltas.csv'}

y_multiples = { 'target_HY3': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3.csv',
                'target_HY4': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4.csv',
                'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43.csv',
                'target_MCID': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID.csv'}

csv_output_paths = {'UPDRS':{'FULL_SET':{'target_HY3': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY3/FULL_SET/', 
                                            'target_HY4': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY4/FULL_SET/', 
                                            'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY43/FULL_SET/', 
                                            'target_MCID': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/MCID/FULL_SET/'},

                                'FEATURE_SELECTION':{'target_HY3': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY3/FEATURE_SELECTION/', 
                                                        'target_HY4': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY4/FEATURE_SELECTION/', 
                                                        'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY43/FEATURE_SELECTION/', 
                                                        'target_MCID': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/MCID/FEATURE_SELECTION/'}},

                    'FULL':{'FULL_SET':{'target_HY3': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY3/FULL_SET/', 
                                            'target_HY4': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY4/FULL_SET/', 
                                            'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY43/FULL_SET/', 
                                            'target_MCID': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/MCID/FULL_SET/'},
                                
                                'FEATURE_SELECTION':{'target_HY3': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY3/FEATURE_SELECTION/', 
                                                        'target_HY4': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY4/FEATURE_SELECTION/', 
                                                        'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY43/FEATURE_SELECTION/', 
                                                        'target_MCID': project_root/'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/MCID/FEATURE_SELECTION/'}}}

dominios_full = {
    'X_STATS': {
        'full': full_domain['SC_data'] + full_domain['M_data_STATS'] + full_domain['NM_data_STATS'],
        'motor': full_domain['M_data_STATS'],
        'non_motor': full_domain['NM_data_STATS']
    },

    'X_V06+STATS': {
        'full': full_domain['SC_data']
                + full_domain['M_data_V06'] + full_domain['M_data_STATS']
                + full_domain['NM_data_V06'] + full_domain['NM_data_STATS'],
        'motor': full_domain['M_data_V06'] + full_domain['M_data_STATS'],
        'non_motor': full_domain['NM_data_V06'] + full_domain['NM_data_STATS']
    },

    'X_V06+DELTA': {
        'full': full_domain['SC_data']
                + full_domain['M_data_V06'] + full_domain['M_data_DELTA']
                + full_domain['NM_data_V06'] + full_domain['NM_data_DELTA'],
        'motor': full_domain['M_data_V06'] + full_domain['M_data_DELTA'],
        'non_motor': full_domain['NM_data_V06'] + full_domain['NM_data_DELTA']
    }
}

dominios_updrs = {
    'X_STATS': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS']
                + updrs_domain['UPDRS_NO_MOTOR_STATS']
                + updrs_domain['UPDRS_MOTOR_STATS'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_STATS'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_STATS'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_STATS']
    },

    'X_V06+STATS': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS']
                + updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_STATS']
                + updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_STATS'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_STATS'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_STATS']
    },

    'X_V06+DELTA': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_delta']
                + updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_delta']
                + updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_delta'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_delta'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_delta'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_delta']
    }
}



classification_models = {

    "decision_tree": DecisionTreeClassifier(random_state=42),

    "random_forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),

    "extra_trees": ExtraTreesClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
        tree_method="hist",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ),


    "adaboost": AdaBoostClassifier(
        algorithm="SAMME", 
        random_state=42
    ),

    "svm": SVC(
        kernel="rbf",
        probability=True,  
        random_state=42
    ),

    "logistic_regression": LogisticRegression(
        random_state=42,
        n_jobs=-1,
        max_iter=10000,
        class_weight="balanced"
    ),

    "knn": KNeighborsClassifier(
        n_jobs=-1
    ),

    "gaussian_nb": GaussianNB()
}

# ANALYSIS DEFAULT MODELS

In [2]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd


def evaluate_models_10x10_oof_and_test(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    scaler = StandardScaler(),
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
):
    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()
    classes = np.unique(y)

    def build_pipeline(estimator):
        return Pipeline([
            ("scaler", scaler),
            ("model", clone(estimator)),
        ])

    def compute_metrics(y_true, y_pred, y_proba):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "AUC_macro": roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"),
            "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "AUC_weighted": roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted"),
        }

    def summarize(metrics_list, suffix):
        df = pd.DataFrame(metrics_list)
        mean = df.mean()
        std = df.std(ddof=1)
        return {
            f"{col}_{suffix}": f"{mean[col]:.{decimals}f} ± {std[col]:.{decimals}f}"
            for col in df.columns
        }


    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []

    for model_name, estimator in models.items():
        print(f"Evaluating {model_name}...")

        test_metrics_all = []
        cv_metrics_all = []

        for train_idx, test_idx in outer.split(X, y):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            inner = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=random_state
            )

            oof_pred = np.zeros(len(y_train), dtype=y_train.dtype)
            oof_proba = np.zeros((len(y_train), len(classes)))

            for tr_idx, val_idx in inner.split(X_train, y_train):
                X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
                X_val = X_train[val_idx]

                model = build_pipeline(estimator)
                model.fit(X_tr, y_tr)

                oof_pred[val_idx] = model.predict(X_val)

                fold_proba = model.predict_proba(X_val)
                fold_classes = model.named_steps["model"].classes_

                aligned_proba = np.zeros((len(val_idx), len(classes)))
                for j, cls in enumerate(fold_classes):
                    aligned_proba[:, np.where(classes == cls)[0][0]] = fold_proba[:, j]

                oof_proba[val_idx] = aligned_proba

            cv_metrics_all.append(compute_metrics(y_train, oof_pred, oof_proba))

            model_full = build_pipeline(estimator)
            model_full.fit(X_train, y_train)

            test_pred = model_full.predict(X_test)
            test_proba_raw = model_full.predict_proba(X_test)
            test_classes = model_full.named_steps["model"].classes_

            test_proba = np.zeros((len(y_test), len(classes)))
            for j, cls in enumerate(test_classes):
                test_proba[:, np.where(classes == cls)[0][0]] = test_proba_raw[:, j]

            test_metrics_all.append(compute_metrics(y_test, test_pred, test_proba))

        test_summary_rows.append(
            pd.Series(summarize(test_metrics_all, "Testing"), name=model_name)
        )
        cv_summary_rows.append(
            pd.Series(summarize(cv_metrics_all, "CV"), name=model_name)
        )

    df_test_summary = pd.DataFrame(test_summary_rows)[[
        "Accuracy_Testing",
        "Precision_macro_Testing", "Recall_macro_Testing", "F1_macro_Testing", "AUC_macro_Testing",
        "Precision_weighted_Testing", "Recall_weighted_Testing", "F1_weighted_Testing", "AUC_weighted_Testing"
    ]]

    df_cv_summary = pd.DataFrame(cv_summary_rows)[[
        "Accuracy_CV",
        "Precision_macro_CV", "Recall_macro_CV", "F1_macro_CV", "AUC_macro_CV",
        "Precision_weighted_CV", "Recall_weighted_CV", "F1_weighted_CV", "AUC_weighted_CV"
    ]]

    df_final_summary = pd.concat([df_test_summary, df_cv_summary], axis=1)
    return df_final_summary

In [3]:
for val in dominios_updrs:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY3', 'target_HY4','target_HY43','target_MCID']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in dominios_updrs[val]:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_updrs[val][val2]))
            X_sub = X[dominios_updrs[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            for scaler in [StandardScaler(), MinMaxScaler()]:
                print(f"      Evaluating with scaler: {scaler.__class__.__name__}")
                df=evaluate_models_10x10_oof_and_test(
                                        X_df=X_sub, 
                                        y_df=y,
                                        scaler=scaler,
                                        models=classification_models, 
                                        random_state=42
                                    )
                print(f"      Saving results to: {csv_output_paths['UPDRS']['FULL_SET'][target]/f'{val}_{val2}_{scaler.__class__.__name__}.csv'}")
                df.to_csv(csv_output_paths['UPDRS']['FULL_SET'][target]/f"{val}_{val2}_{scaler.__class__.__name__}_{target}.csv")
                

Evaluating domain: X_STATS
  Target: target_HY3, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 301
      Data shape: (909, 301)
      Evaluating with scaler: StandardScaler
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
      Saving results to: /home/fsc/Desktop/PD_progression_ppmi/SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY3/FULL_SET/X_STATS_full_StandardScaler.csv
      Evaluating with scaler: MinMaxScaler
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
      Saving results to: /home/fsc/Desktop/PD_progression_ppmi/SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/

In [4]:
for val in dominios_full:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY3', 'target_HY4','target_HY43','target_MCID']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in dominios_full[val]:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_full[val][val2]))
            X_sub = X[dominios_full[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            for scaler in [StandardScaler(), MinMaxScaler()]:
                print(f"      Evaluating with scaler: {scaler.__class__.__name__}")
                df=evaluate_models_10x10_oof_and_test(
                                        X_df=X_sub, 
                                        y_df=y,
                                        scaler=scaler, 
                                        models=classification_models, 
                                        random_state=42
                                    )
                print(f"      Saving results to: {csv_output_paths['FULL']['FULL_SET'][target]/f'{val}_{val2}_{scaler.__class__.__name__}.csv'}")
                df.to_csv(csv_output_paths['FULL']['FULL_SET'][target]/f"{val}_{val2}_{scaler.__class__.__name__}_{target}.csv")

Evaluating domain: X_STATS
  Target: target_HY3, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 931
      Data shape: (909, 931)
      Evaluating with scaler: StandardScaler
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
      Saving results to: /home/fsc/Desktop/PD_progression_ppmi/SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY3/FULL_SET/X_STATS_full_StandardScaler.csv
      Evaluating with scaler: MinMaxScaler
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
      Saving results to: /home/fsc/Desktop/PD_progression_ppmi/SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/C

# AGNOSTIC FEATURE SELECTION

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, chi2
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


# =========================================================
# Feature selectors
# =========================================================

class SpearmanSULOVSelector(BaseEstimator, TransformerMixin):
    """
    Elimina variables altamente correlacionadas usando correlación de Spearman.
    Entre dos variables correlacionadas, conserva la que tenga mayor mutual information con y.
    """
    def __init__(self, threshold=0.9, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.feature_names_in_ = None
        self.mi_scores_ = None
        self.vars_to_drop_ = None
        self.selected_features_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        mi = mutual_info_classif(X, y, random_state=self.random_state)
        self.mi_scores_ = pd.Series(mi, index=X.columns)

        corr_df = X.corr(method="spearman").abs()
        upper = corr_df.where(np.triu(np.ones(corr_df.shape), k=1).astype(bool))

        vars_to_drop = set()

        for col in upper.columns:
            correlated_features = upper.index[upper[col] > self.threshold].tolist()

            for row_feature in correlated_features:
                if row_feature in vars_to_drop or col in vars_to_drop:
                    continue

                if self.mi_scores_[row_feature] <= self.mi_scores_[col]:
                    vars_to_drop.add(row_feature)
                else:
                    vars_to_drop.add(col)

        self.vars_to_drop_ = list(vars_to_drop)
        self.selected_features_ = [c for c in X.columns if c not in self.vars_to_drop_]
        self.n_features_selected_ = len(self.selected_features_)

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names_in_)
        return X.drop(columns=self.vars_to_drop_, errors="ignore")


class SpearmanCorrelationDiscard(BaseEstimator, TransformerMixin):
    """
    Elimina variables altamente correlacionadas según Spearman,
    conservando la que tenga mayor correlación absoluta con el target.
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.vars_to_drop_ = None
        self.feature_names_in_ = None

    def fit(self, X, y=None):
        if y is None:
            raise ValueError("SpearmanCorrelationDiscard requiere y en fit().")

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        data = X.copy()
        data["_target_"] = y

        corr_df = data.corr(method="spearman")

        mask = np.tril(np.ones(corr_df.shape), k=-1).astype(bool)

        corr_long = (
            corr_df.where(mask)
            .stack()
            .reset_index()
            .rename(columns={"level_0": "V1", "level_1": "V2", 0: "CORR"})
        )

        corr_long = corr_long[
            (corr_long["V1"] != "_target_") & (corr_long["V2"] != "_target_")
        ].copy()

        target_corr = corr_df["_target_"]

        corr_long["V1target"] = corr_long["V1"].map(target_corr)
        corr_long["V2target"] = corr_long["V2"].map(target_corr)

        corr_long["WORST_VAR"] = np.where(
            abs(corr_long["V1target"]) <= abs(corr_long["V2target"]),
            corr_long["V1"],
            corr_long["V2"]
        )

        discard_corr_long = corr_long.loc[corr_long["CORR"].abs() > self.threshold]
        self.vars_to_drop_ = list(set(discard_corr_long["WORST_VAR"]))

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.vars_to_drop_, errors="ignore")

        X_df = pd.DataFrame(X, columns=self.feature_names_in_)
        X_df = X_df.drop(columns=self.vars_to_drop_, errors="ignore")
        return X_df


class MIThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.support_ = None
        self.mi_scores_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)
        self.support_ = self.mi_scores_ >= self.threshold

        if not np.any(self.support_):
            self.support_[np.argmax(self.mi_scores_)] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


class Chi2ThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.0):
        self.threshold = threshold
        self.support_ = None
        self.chi2_scores_ = None
        self.pvalues_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.chi2_scores_, self.pvalues_ = chi2(X_fit, y)
        self.support_ = self.chi2_scores_ >= self.threshold

        if not np.any(self.support_):
            self.support_[np.argmax(self.chi2_scores_)] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


class Chi2MIUnionTopKSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona la unión entre:
    - top_k variables según chi2
    - top_k variables según mutual information

    Nota:
    - chi2 requiere valores no negativos, por eso este selector debe usarse
      después de un MinMaxScaler.
    """
    def __init__(self, top_k=100, random_state=42):
        self.top_k = top_k
        self.random_state = random_state
        self.feature_names_in_ = None
        self.chi2_scores_ = None
        self.mi_scores_ = None
        self.selected_features_ = None
        self.support_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        n_features = X_fit.shape[1]
        k = min(self.top_k, n_features)

        self.chi2_scores_, _ = chi2(X_fit, y)
        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)

        top_chi2_idx = np.argsort(self.chi2_scores_)[::-1][:k]
        top_mi_idx = np.argsort(self.mi_scores_)[::-1][:k]

        selected_idx = np.union1d(top_chi2_idx, top_mi_idx)

        self.support_ = np.zeros(n_features, dtype=bool)
        self.support_[selected_idx] = True
        self.n_features_selected_ = int(self.support_.sum())

        if self.feature_names_in_ is not None:
            self.selected_features_ = [self.feature_names_in_[i] for i in selected_idx]
        else:
            self.selected_features_ = selected_idx.tolist()

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


# =========================================================
# Helpers
# =========================================================

def build_pipeline(
    estimator,
    selector_name,
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
    chi2_mi_top_k=100,
    random_state=42,
):
    if selector_name == "spearman_corr":
        return Pipeline([
            ("selector", SpearmanCorrelationDiscard(threshold=spearman_threshold)),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "spearman_sulov":
        return Pipeline([
            ("selector", SpearmanSULOVSelector(
                threshold=spearman_threshold,
                random_state=random_state
            )),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "mutual_info":
        return Pipeline([
            ("selector", MIThresholdSelector(
                threshold=mi_threshold,
                random_state=random_state
            )),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "chi2":
        return Pipeline([
            ("minmax_before_chi2", MinMaxScaler()),
            ("selector", Chi2ThresholdSelector(threshold=chi2_threshold)),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "chi2_mi_union_topk":
        return Pipeline([
            ("minmax_before_union", MinMaxScaler()),
            ("selector", Chi2MIUnionTopKSelector(
                top_k=chi2_mi_top_k,
                random_state=random_state
            )),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    else:
        raise ValueError(f"Selector no soportado: {selector_name}")


def safe_multiclass_auc(y_true, y_proba, average="macro"):
    try:
        present_classes = np.unique(y_true)
        if len(present_classes) < 2:
            return np.nan

        y_proba_used = y_proba[:, present_classes]
        mapper = {cls: i for i, cls in enumerate(present_classes)}
        y_true_mapped = np.array([mapper[v] for v in y_true])

        return roc_auc_score(
            y_true_mapped,
            y_proba_used,
            multi_class="ovr",
            average=average
        )
    except Exception:
        return np.nan


def compute_metrics(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "AUC_macro": safe_multiclass_auc(y_true, y_proba, average="macro"),
        "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "AUC_weighted": safe_multiclass_auc(y_true, y_proba, average="weighted"),
    }


def summarize(metrics_list, suffix="CV", decimals=4):
    df = pd.DataFrame(metrics_list)
    mean = df.mean(numeric_only=True)
    std = df.std(ddof=1, numeric_only=True)

    return {
        f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
        for c in mean.index
    }


def _predict_proba_aligned(pipe, X, classes_global):
    proba = pipe.predict_proba(X)
    model_classes = pipe.named_steps["model"].classes_

    aligned = np.zeros((len(X), len(classes_global)), dtype=float)
    class_to_global_idx = {c: i for i, c in enumerate(classes_global)}

    for local_idx, cls in enumerate(model_classes):
        aligned[:, class_to_global_idx[cls]] = proba[:, local_idx]

    return aligned


# =========================================================
# Función principal
# =========================================================

def evaluate_models_oof_and_test_with_fs(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
    selectors=None,
    spearman_threshold: float = 0.9,
    mi_threshold: float = 0.01,
    chi2_threshold: float = 3.84,
    chi2_mi_top_k: int = 100,
):
    if selectors is None:
        selectors = [
            "spearman_corr",
            "mutual_info",
            "spearman_sulov",
            "chi2",
            "chi2_mi_union_topk",
        ]

    X = X_df.copy()
    y = y_df.iloc[:, 0].to_numpy()

    models = {
        name: model
        for name, model in models.items()
        if hasattr(model, "predict_proba")
    }

    if len(models) == 0:
        raise ValueError("Ningún modelo tiene el método predict_proba().")

    classes_global = np.unique(y)
    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    all_rows = []

    for model_name, estimator in models.items():
        for selector_name in selectors:
            print(f"Evaluating {model_name} + {selector_name}...")

            test_metrics_all = []
            cv_metrics_all = []

            for train_idx, test_idx in outer.split(X, y):
                X_train = X.iloc[train_idx].copy()
                X_test = X.iloc[test_idx].copy()
                y_train = y[train_idx]
                y_test = y[test_idx]

                _, class_counts = np.unique(y_train, return_counts=True)
                min_class_count = class_counts.min()
                effective_inner_splits = min(inner_splits, min_class_count)

                if effective_inner_splits < 2:
                    raise ValueError(
                        "No hay suficientes muestras por clase para realizar StratifiedKFold."
                    )

                inner = StratifiedKFold(
                    n_splits=effective_inner_splits,
                    shuffle=True,
                    random_state=random_state
                )

                oof_pred = np.empty(len(y_train), dtype=y_train.dtype)
                oof_proba = np.zeros((len(y_train), len(classes_global)), dtype=float)

                for tr_idx, val_idx in inner.split(X_train, y_train):
                    X_tr = X_train.iloc[tr_idx].copy()
                    X_val = X_train.iloc[val_idx].copy()
                    y_tr = y_train[tr_idx]

                    pipe = build_pipeline(
                        estimator=estimator,
                        selector_name=selector_name,
                        spearman_threshold=spearman_threshold,
                        mi_threshold=mi_threshold,
                        chi2_threshold=chi2_threshold,
                        chi2_mi_top_k=chi2_mi_top_k,
                        random_state=random_state,
                    )

                    pipe.fit(X_tr, y_tr)

                    oof_pred[val_idx] = pipe.predict(X_val)
                    oof_proba[val_idx] = _predict_proba_aligned(pipe, X_val, classes_global)

                cv_metrics_all.append(
                    compute_metrics(
                        y_true=y_train,
                        y_pred=oof_pred,
                        y_proba=oof_proba
                    )
                )

                pipe_full = build_pipeline(
                    estimator=estimator,
                    selector_name=selector_name,
                    spearman_threshold=spearman_threshold,
                    mi_threshold=mi_threshold,
                    chi2_threshold=chi2_threshold,
                    chi2_mi_top_k=chi2_mi_top_k,
                    random_state=random_state,
                )

                pipe_full.fit(X_train, y_train)

                test_pred = pipe_full.predict(X_test)
                test_proba = _predict_proba_aligned(pipe_full, X_test, classes_global)

                test_metrics_all.append(
                    compute_metrics(
                        y_true=y_test,
                        y_pred=test_pred,
                        y_proba=test_proba
                    )
                )

            row = {
                "F1_macro_CV": summarize(cv_metrics_all, suffix="CV", decimals=decimals)["F1_macro_CV"],
                "F1_macro_Testing": summarize(test_metrics_all, suffix="Testing", decimals=decimals)["F1_macro_Testing"],
                "Model": model_name,
                "Feature_Selection": selector_name,
            }

            row.update(summarize(test_metrics_all, suffix="Testing", decimals=decimals))
            row.update(summarize(cv_metrics_all, suffix="CV", decimals=decimals))

            all_rows.append(row)

    df_final_summary = pd.DataFrame(all_rows)[[
        "Model",
        "Feature_Selection",
        "F1_macro_CV",
        "F1_macro_Testing",
        "Accuracy_Testing",
        "Precision_macro_Testing",
        "Recall_macro_Testing",
        "AUC_macro_Testing",
        "Precision_weighted_Testing",
        "Recall_weighted_Testing",
        "F1_weighted_Testing",
        "AUC_weighted_Testing",
        "Accuracy_CV",
        "Precision_macro_CV",
        "Recall_macro_CV",
        "AUC_macro_CV",
        "Precision_weighted_CV",
        "Recall_weighted_CV",
        "F1_weighted_CV",
        "AUC_weighted_CV",
    ]]

    return df_final_summary



In [ ]:
for val in dominios_updrs:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY3', 'target_HY4','target_HY43','target_MCID']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in ['full', 'motor']:
            if val2 == 'motor':
                dominios=dominios_updrs[val]['examen_motor']+dominios_updrs[val]['impacto_motor']
            else:
                dominios=dominios_updrs[val][val2]
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios))
            X_sub = X[dominios]
            print(f"      Data shape: {X_sub.shape}")
            print(f"      Evaluating...")
            df=evaluate_models_oof_and_test_with_fs(
                                                    X_df=X_sub,
                                                    y_df=y,
                                                    models=classification_models)
                
            print(f"      Saving results to: {csv_output_paths['UPDRS']['FEATURE_SELECTION'][target]/f'{val}_{val2}_.csv'}")
            df.to_csv(csv_output_paths['UPDRS']['FEATURE_SELECTION'][target]/f"{val}_{val2}_feature_selection_{target}.csv")

In [ ]:
for val in dominios_full:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY3', 'target_HY4','target_HY43','target_MCID']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in ['full', 'motor']:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_full[val][val2]))
            X_sub = X[dominios_full[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            print(f"      Evaluating...")
            df=evaluate_models_oof_and_test_with_fs(
                                                    X_df=X_sub,
                                                    y_df=y,
                                                    models=classification_models)
            
            print(f"      Saving results to: {csv_output_paths['FULL']['FEATURE_SELECTION'][target]/f'{val}_{val2}_.csv'}")
            df.to_csv(csv_output_paths['FULL']['FEATURE_SELECTION'][target]/f"{val}_{val2}_feature_selection_{target}.csv")  

Evaluating domain: X_STATS
  Target: target_HY3, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 931
      Data shape: (909, 931)
      Evaluating...


NameError: name 'evaluate_models_oof_and_test_with_fs' is not defined

# BAYESIAN OPTIMIZATION
- FEATURE SELECTION: chi2_mi_union_topk
- HYPERPARAMETER TURNING

In [21]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, chi2
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical


class Chi2MIUnionTopKSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona la unión entre:
    - top_k variables según chi2
    - top_k variables según mutual information

    Nota:
    - chi2 requiere valores no negativos, por eso este selector debe usarse
      después de un MinMaxScaler.
    """
    def __init__(self, top_k=100, random_state=42):
        self.top_k = top_k
        self.random_state = random_state
        self.feature_names_in_ = None
        self.chi2_scores_ = None
        self.mi_scores_ = None
        self.selected_features_ = None
        self.support_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        n_features = X_fit.shape[1]
        k = min(self.top_k, n_features)

        self.chi2_scores_, _ = chi2(X_fit, y)
        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)

        top_chi2_idx = np.argsort(self.chi2_scores_)[::-1][:k]
        top_mi_idx = np.argsort(self.mi_scores_)[::-1][:k]

        selected_idx = np.union1d(top_chi2_idx, top_mi_idx)

        self.support_ = np.zeros(n_features, dtype=bool)
        self.support_[selected_idx] = True
        self.n_features_selected_ = int(self.support_.sum())

        if self.feature_names_in_ is not None:
            self.selected_features_ = [self.feature_names_in_[i] for i in selected_idx]
        else:
            self.selected_features_ = selected_idx.tolist()

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]
        return X[:, self.support_]


def evaluate_models_nested_bayes(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 5,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
    n_iter_search: int = 40,
    n_jobs_search: int = -1,
    selector_name: str = None,
    chi2_mi_top_k: int = 100,
):
    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()
    classes = np.unique(y)

    def build_pipeline(estimator, selector_name=None):
        if selector_name is None:
            return Pipeline([
                ("scaler", MinMaxScaler()),
                ("model", clone(estimator)),
            ])

        elif selector_name == "chi2_mi_union_topk":
            return Pipeline([
                ("minmax_before_union", MinMaxScaler()),
                ("selector", Chi2MIUnionTopKSelector(
                    top_k=chi2_mi_top_k,
                    random_state=random_state
                )),
                ("scaler", MinMaxScaler()),
                ("model", clone(estimator)),
            ])

        else:
            raise ValueError(f"Selector no soportado: {selector_name}")

    def get_search_spaces(model_name):
        spaces = {
            "decision_tree": {
                "model__max_depth": Integer(2, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__criterion": Categorical(["gini", "entropy"]),
                "model__max_features": Categorical([None, "sqrt"]),
                "model__class_weight": Categorical([None, "balanced"]),
            },

            "random_forest": {
                "model__n_estimators": Integer(200, 800),
                "model__max_depth": Integer(4, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__max_features": Categorical(["sqrt", "log2"]),
                "model__bootstrap": Categorical([True]),
                "model__class_weight": Categorical([None, "balanced", "balanced_subsample"]),
            },

            "extra_trees": {
                "model__n_estimators": Integer(200, 800),
                "model__max_depth": Integer(4, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__max_features": Categorical(["sqrt", "log2"]),
                "model__class_weight": Categorical([None, "balanced", "balanced_subsample"]),
            },

            "xgboost": {
                "model__n_estimators": Integer(200, 600),
                "model__max_depth": Integer(3, 10),
                "model__learning_rate": Real(0.01, 0.3, prior="log-uniform"),
                "model__subsample": Real(0.6, 1.0),
                "model__colsample_bytree": Real(0.6, 1.0),
                "model__min_child_weight": Integer(1, 10),
                "model__gamma": Real(1e-8, 5.0, prior="log-uniform"),
                "model__reg_alpha": Real(1e-8, 5.0, prior="log-uniform"),
                "model__reg_lambda": Real(1e-8, 5.0, prior="log-uniform"),
            },

            "adaboost": {
                "model__n_estimators": Integer(50, 500),
                "model__learning_rate": Real(0.01, 1.0, prior="log-uniform"),
            },

            "svm": {
                "model__C": Real(1e-3, 1e3, prior="log-uniform"),
                "model__gamma": Real(1e-5, 1.0, prior="log-uniform"),
                "model__kernel": Categorical(["rbf"]),
                "model__class_weight": Categorical([None, "balanced"]),
            },

            "logistic_regression": {
                "model__C": Real(1e-4, 1e2, prior="log-uniform"),
                "model__solver": Categorical(["lbfgs", "saga"]),
                "model__penalty": Categorical(["l2"]),
                "model__class_weight": Categorical([None, "balanced"]),
            },

            "knn": {
                "model__n_neighbors": Integer(3, 51),
                "model__weights": Categorical(["uniform", "distance"]),
                "model__p": Integer(1, 2),
            },

            "gaussian_nb": {
                "model__var_smoothing": Real(1e-10, 1e-6, prior="log-uniform"),
            },
        }

        if model_name not in spaces:
            raise ValueError(f"No search space definido para {model_name}")

        return spaces[model_name]

    def compute_metrics(y_true, y_pred, y_proba):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "AUC_macro": roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"),
            "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "AUC_weighted": roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted"),
        }

    def summarize(metrics_list, suffix):
        df = pd.DataFrame(metrics_list)
        mean = df.mean(numeric_only=True)
        std = df.std(ddof=1, numeric_only=True)
        return {
            f"{col}_{suffix}": f"{mean[col]:.{decimals}f} ± {std[col]:.{decimals}f}"
            for col in df.columns
        }

    class TqdmBayesCallback:
        """
        Callback para actualizar una barra tqdm en cada iteración
        del BayesSearchCV.
        """
        def __init__(self, pbar):
            self.pbar = pbar

        def __call__(self, res):
            best_cv = -np.min(res.func_vals)
            self.pbar.update(1)
            self.pbar.set_postfix({
                "best_inner_cv_f1": f"{best_cv:.4f}"
            })
            return False

    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []
    best_params_rows = []

    for model_idx, (model_name, estimator) in enumerate(models.items(), start=1):
        print(f"\nEvaluating {model_name} with Bayesian Search...")

        test_metrics_all = []
        cv_metrics_all = []
        best_params_per_outer_fold = []

        search_spaces = get_search_spaces(model_name)

        outer_pbar = tqdm(
            total=outer_splits,
            desc=f"[{model_idx}/{len(models)}] {model_name} OUTER",
            position=0,
            leave=True
        )

        for fold_id, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            inner = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=random_state
            )

            base_pipeline = build_pipeline(estimator, selector_name=selector_name)

            inner_pbar = tqdm(
                total=n_iter_search,
                desc=f"Outer fold {fold_id}/{outer_splits} | Bayes {n_iter_search} iters | inner_cv={inner_splits}",
                position=1,
                leave=False
            )

            opt = BayesSearchCV(
                estimator=base_pipeline,
                search_spaces=search_spaces,
                n_iter=n_iter_search,
                scoring="f1_macro",
                cv=inner,
                n_jobs=n_jobs_search,
                refit=True,
                random_state=random_state,
                verbose=0,
            )

            callback = TqdmBayesCallback(inner_pbar)
            opt.fit(X_train, y_train, callback=callback)

            inner_pbar.close()

            best_model = opt.best_estimator_
            best_params_per_outer_fold.append(opt.best_params_)

            # Guardar info del selector si existe
            if selector_name == "chi2_mi_union_topk":
                selector = best_model.named_steps["selector"]
                best_params_per_outer_fold[-1]["selected_features_count"] = selector.n_features_selected_
                best_params_per_outer_fold[-1]["selected_features"] = selector.selected_features_

            cv_metrics_all.append({
                "F1_macro": opt.best_score_,
            })

            test_pred = best_model.predict(X_test)
            test_proba_raw = best_model.predict_proba(X_test)

            test_classes = best_model.named_steps["model"].classes_
            test_proba = np.zeros((len(y_test), len(classes)))

            for j, cls in enumerate(test_classes):
                test_proba[:, np.where(classes == cls)[0][0]] = test_proba_raw[:, j]

            outer_metrics = compute_metrics(y_test, test_pred, test_proba)
            test_metrics_all.append(outer_metrics)

            avg_outer_f1 = np.mean([m["F1_macro"] for m in test_metrics_all])
            avg_inner_cv_f1 = np.mean([m["F1_macro"] for m in cv_metrics_all])

            outer_pbar.update(1)
            outer_pbar.set_postfix({
                "fold": f"{fold_id}/{outer_splits}",
                "avg_outer_f1": f"{avg_outer_f1:.4f}",
                "avg_inner_cv_f1": f"{avg_inner_cv_f1:.4f}",
            })

        outer_pbar.close()

        test_summary_rows.append(
            pd.Series(summarize(test_metrics_all, "Testing"), name=model_name)
        )

        cv_summary_rows.append(
            pd.Series(summarize(cv_metrics_all, "CV"), name=model_name)
        )

        best_params_rows.append(
            pd.Series({"Best_Params_Outer_Folds": best_params_per_outer_fold}, name=model_name)
        )

    df_test_summary = pd.DataFrame(test_summary_rows)[[
        "Accuracy_Testing",
        "Precision_macro_Testing",
        "Recall_macro_Testing",
        "F1_macro_Testing",
        "AUC_macro_Testing",
        "Precision_weighted_Testing",
        "Recall_weighted_Testing",
        "F1_weighted_Testing",
        "AUC_weighted_Testing"
    ]]

    df_cv_summary = pd.DataFrame(cv_summary_rows)[[
        "F1_macro_CV"
    ]]

    df_best_params = pd.DataFrame(best_params_rows)

    df_final_summary = pd.concat(
        [df_test_summary, df_cv_summary, df_best_params],
        axis=1
    )

    return df_final_summary

In [ ]:
for val in dominios_updrs:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43','target_HY3']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in ['full', 'motor']:
            if val2 == 'motor':
                dominios=dominios_updrs[val]['examen_motor']+dominios_updrs[val]['impacto_motor']
            else:
                dominios=dominios_updrs[val][val2]
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios))
            X_sub = X[dominios]
            print(f"      Data shape: {X_sub.shape}")
            for selector in ['chi2_mi_union_topk', None]:
                print(f"      Using selector: {selector}")
                print(f"      Evaluating...")
                df=evaluate_models_nested_bayes(
                                                X_df=X_sub,
                                                y_df=y,
                                                models=classification_models,
                                                selector_name=selector,   
                                                chi2_mi_top_k=100,                    
                                                outer_splits=10,
                                                inner_splits=10,
                                                n_iter_search=40,)
                
                print(f"      Saving results to: {csv_output_paths['UPDRS']['FEATURE_SELECTION'][target]/f"{val}_{val2}_FS_{selector}_{target}.csv"}")
                df.to_csv(csv_output_paths['UPDRS']['FEATURE_SELECTION'][target]/f"{val}_{val2}_FS_{selector}_{target}_OPT_BAYESIAN.csv")
               
                

Evaluating domain: X_STATS
  Target: target_HY43, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 301
      Data shape: (909, 301)
      Using selector: chi2_mi_union_topk
      Evaluating...

Evaluating decision_tree with Bayesian Search...


[1/9] decision_tree OUTER:   0%|          | 0/10 [00:00<?, ?it/s]

Outer fold 1/10 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Outer fold 2/10 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

...

Outer fold 4/10 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

Outer fold 5/10 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

Outer fold 6/10 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

Outer fold 7/10 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

Outer fold 8/10 | Bayes 40 iters | inner_cv=10:   0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
for val in dominios_full:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43','target_HY3']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in ['full', 'motor']:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_full[val][val2]))
            X_sub = X[dominios_full[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            
            for selector in ['chi2_mi_union_topk', None]:
                print(f"      Using selector: {selector}")
                print(f"      Evaluating...")
                df=evaluate_models_nested_bayes(
                                                X_df=X_sub,
                                                y_df=y,
                                                models=classification_models,
                                                selector_name=selector,   
                                                chi2_mi_top_k=100,                    
                                                outer_splits=10,
                                                inner_splits=10,
                                                n_iter_search=40,)
                print(f"      Saving results to: {csv_output_paths['FULL']['FEATURE_SELECTION'][target]/f"{val}_{val2}_FS_{selector}_{target}.csv"}")
                df.to_csv(csv_output_paths['FULL']['FEATURE_SELECTION'][target]/f"{val}_{val2}_FS_{selector}_{target}_OPT_BAYESIAN.csv")  

# OVERSAMPLING METHOD
- RANDOM OVERSAMPLIG 
- SMOTE LIGHT

In [18]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import  MinMaxScaler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
import numpy as np
import pandas as pd


def evaluate_models_10x10_oof_and_test_SAMPLER(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    scaler=MinMaxScaler(),
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.3,
    sampler: str = "RandomOverSampler",  # "RandomOverSampler" o "SMOTE"
    stategy: dict = None,  # Solo para SMOTE: dict con la estrategia de remuestreo, e.g. {1: 150}
    random_state: int = 42,
    decimals: int = 4,
):
    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()
    classes = np.unique(y)

    def build_pipeline(estimator):
        if sampler == "RandomOverSampler":
            return ImbPipeline([
                ("scaler", scaler),
                ("sampler", RandomOverSampler(
                    sampling_strategy=stategy,
                    random_state=random_state
                )),
                ("model", clone(estimator)),
            ])
        elif sampler == "SMOTE":
            return ImbPipeline([
                ("scaler", scaler),
                ("sampler", SMOTE(
                    sampling_strategy=stategy,
                    k_neighbors=2,
                    random_state=random_state
                )),
                ("model", clone(estimator)),
            ])
        else:
            raise ValueError("sampler debe ser 'RandomOverSampler' o 'SMOTE'")

    def compute_metrics(y_true, y_pred, y_proba):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "AUC_macro": roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"),
            "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "AUC_weighted": roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted"),
        }

    def summarize(metrics_list, suffix):
        df = pd.DataFrame(metrics_list)
        mean = df.mean()
        std = df.std(ddof=1)
        return {
            f"{col}_{suffix}": f"{mean[col]:.{decimals}f} ± {std[col]:.{decimals}f}"
            for col in df.columns
        }

    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []

    for model_name, estimator in models.items():
        print(f"Evaluating {model_name}...")

        test_metrics_all = []
        cv_metrics_all = []

        for train_idx, test_idx in outer.split(X, y):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            inner = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=random_state
            )

            oof_pred = np.zeros(len(y_train), dtype=y_train.dtype)
            oof_proba = np.zeros((len(y_train), len(classes)))

            for tr_idx, val_idx in inner.split(X_train, y_train):
                X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
                X_val = X_train[val_idx]

                model = build_pipeline(estimator)
                model.fit(X_tr, y_tr)

                oof_pred[val_idx] = model.predict(X_val)

                fold_proba = model.predict_proba(X_val)
                fold_classes = model.named_steps["model"].classes_

                aligned_proba = np.zeros((len(val_idx), len(classes)))
                for j, cls in enumerate(fold_classes):
                    aligned_proba[:, np.where(classes == cls)[0][0]] = fold_proba[:, j]

                oof_proba[val_idx] = aligned_proba

            cv_metrics_all.append(compute_metrics(y_train, oof_pred, oof_proba))

            model_full = build_pipeline(estimator)
            model_full.fit(X_train, y_train)

            test_pred = model_full.predict(X_test)
            test_proba_raw = model_full.predict_proba(X_test)
            test_classes = model_full.named_steps["model"].classes_

            test_proba = np.zeros((len(y_test), len(classes)))
            for j, cls in enumerate(test_classes):
                test_proba[:, np.where(classes == cls)[0][0]] = test_proba_raw[:, j]

            test_metrics_all.append(compute_metrics(y_test, test_pred, test_proba))

        test_summary_rows.append(
            pd.Series(summarize(test_metrics_all, "Testing"), name=model_name)
        )
        cv_summary_rows.append(
            pd.Series(summarize(cv_metrics_all, "CV"), name=model_name)
        )

    df_test_summary = pd.DataFrame(test_summary_rows)[[
        "Accuracy_Testing",
        "Precision_macro_Testing", "Recall_macro_Testing", "F1_macro_Testing", "AUC_macro_Testing",
        "Precision_weighted_Testing", "Recall_weighted_Testing", "F1_weighted_Testing", "AUC_weighted_Testing"
    ]]

    df_cv_summary = pd.DataFrame(cv_summary_rows)[[
        "Accuracy_CV",
        "Precision_macro_CV", "Recall_macro_CV", "F1_macro_CV", "AUC_macro_CV",
        "Precision_weighted_CV", "Recall_weighted_CV", "F1_weighted_CV", "AUC_weighted_CV"
    ]]

    df_final_summary = pd.concat([df_test_summary, df_cv_summary], axis=1)
    return df_final_summary

In [19]:
for val in dominios_updrs:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43','target_HY3']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")
        if target == 'target_HY43':
                stategy_sampling={1: 150}
        else:
                stategy_sampling={2: 50}
        for val2 in dominios_updrs[val]:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_updrs[val][val2]))
            X_sub = X[dominios_updrs[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            for sampler in ['RandomOverSampler', 'SMOTE']:
                print(f"      Evaluating with sampler: {sampler}")
                df=evaluate_models_10x10_oof_and_test_SAMPLER(
                                        X_df=X_sub, 
                                        y_df=y,
                                        sampler=sampler,
                                        stategy=stategy_sampling,
                                        models=classification_models, 
                                        random_state=42
                                    )
                print(f"      Saving results to: {csv_output_paths['UPDRS']['FULL_SET'][target]/f'{val}_{val2}_{sampler}.csv'}")
                df.to_csv(csv_output_paths['UPDRS']['FULL_SET'][target]/f"{val}_{val2}_{sampler}_{target}.csv")
                

Evaluating domain: X_STATS
  Target: target_HY43, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 301
      Data shape: (909, 301)
      Evaluating with sampler: RandomOverSampler
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
      Saving results to: /home/fsc/Desktop/PD_progression_ppmi/SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY43/FULL_SET/X_STATS_full_RandomOverSampler.csv
      Evaluating with sampler: SMOTE
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
      Saving results to: /home/fsc/Desktop/PD_progression_ppmi/SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESUL

In [20]:
for val in dominios_full:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43','target_HY3']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")
        if target == 'target_HY43':
                stategy_sampling={1: 150}
        else:
                stategy_sampling={2: 50}

        for val2 in dominios_full[val]:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_full[val][val2]))
            X_sub = X[dominios_full[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            for sampler in ['RandomOverSampler', 'SMOTE']:
                print(f"      Evaluating with sampler: {sampler}")
                df=evaluate_models_10x10_oof_and_test_SAMPLER(
                                        X_df=X_sub, 
                                        y_df=y,
                                        sampler=sampler, 
                                        models=classification_models, 
                                        random_state=42,
                                        stategy=stategy_sampling
                                    )
                print(f"      Saving results to: {csv_output_paths['FULL']['FULL_SET'][target]/f'{val}_{val2}_{sampler}.csv'}")
                df.to_csv(csv_output_paths['FULL']['FULL_SET'][target]/f"{val}_{val2}_{sampler}_{target}.csv")

Evaluating domain: X_STATS
  Target: target_HY43, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 931
      Data shape: (909, 931)
      Evaluating with sampler: RandomOverSampler
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
      Saving results to: /home/fsc/Desktop/PD_progression_ppmi/SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY43/FULL_SET/X_STATS_full_RandomOverSampler.csv
      Evaluating with sampler: SMOTE
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
      Saving results to: /home/fsc/Desktop/PD_progression_ppmi/SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULT

# AGNOSTIC FEATURE SELECTION + OVERSAMPLING


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, chi2
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# =========================================================
# Feature selectors
# =========================================================

class SpearmanSULOVSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.feature_names_in_ = None
        self.mi_scores_ = None
        self.vars_to_drop_ = None
        self.selected_features_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        mi = mutual_info_classif(X, y, random_state=self.random_state)
        self.mi_scores_ = pd.Series(mi, index=X.columns)

        corr_df = X.corr(method="spearman").abs()
        upper = corr_df.where(np.triu(np.ones(corr_df.shape), k=1).astype(bool))

        vars_to_drop = set()

        for col in upper.columns:
            correlated_features = upper.index[upper[col] > self.threshold].tolist()

            for row_feature in correlated_features:
                if row_feature in vars_to_drop or col in vars_to_drop:
                    continue

                if self.mi_scores_[row_feature] <= self.mi_scores_[col]:
                    vars_to_drop.add(row_feature)
                else:
                    vars_to_drop.add(col)

        self.vars_to_drop_ = list(vars_to_drop)
        self.selected_features_ = [c for c in X.columns if c not in self.vars_to_drop_]
        self.n_features_selected_ = len(self.selected_features_)
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names_in_)
        return X.drop(columns=self.vars_to_drop_, errors="ignore")


class SpearmanCorrelationDiscard(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.vars_to_drop_ = None
        self.feature_names_in_ = None

    def fit(self, X, y=None):
        if y is None:
            raise ValueError("SpearmanCorrelationDiscard requiere y en fit().")

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        data = X.copy()
        data["_target_"] = y

        corr_df = data.corr(method="spearman")
        mask = np.tril(np.ones(corr_df.shape), k=-1).astype(bool)

        corr_long = (
            corr_df.where(mask)
            .stack()
            .reset_index()
            .rename(columns={"level_0": "V1", "level_1": "V2", 0: "CORR"})
        )

        corr_long = corr_long[
            (corr_long["V1"] != "_target_") & (corr_long["V2"] != "_target_")
        ].copy()

        target_corr = corr_df["_target_"]

        corr_long["V1target"] = corr_long["V1"].map(target_corr)
        corr_long["V2target"] = corr_long["V2"].map(target_corr)

        corr_long["WORST_VAR"] = np.where(
            abs(corr_long["V1target"]) <= abs(corr_long["V2target"]),
            corr_long["V1"],
            corr_long["V2"]
        )

        discard_corr_long = corr_long.loc[corr_long["CORR"].abs() > self.threshold]
        self.vars_to_drop_ = list(set(discard_corr_long["WORST_VAR"]))

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.vars_to_drop_, errors="ignore")

        X_df = pd.DataFrame(X, columns=self.feature_names_in_)
        return X_df.drop(columns=self.vars_to_drop_, errors="ignore")


class MIThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.support_ = None
        self.mi_scores_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)
        self.support_ = self.mi_scores_ >= self.threshold

        if not np.any(self.support_):
            self.support_[np.argmax(self.mi_scores_)] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]
        return X[:, self.support_]


class Chi2ThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.0):
        self.threshold = threshold
        self.support_ = None
        self.chi2_scores_ = None
        self.pvalues_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.chi2_scores_, self.pvalues_ = chi2(X_fit, y)
        self.support_ = self.chi2_scores_ >= self.threshold

        if not np.any(self.support_):
            self.support_[np.argmax(self.chi2_scores_)] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]
        return X[:, self.support_]


class Chi2MIUnionTopKSelector(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=100, random_state=42):
        self.top_k = top_k
        self.random_state = random_state
        self.feature_names_in_ = None
        self.chi2_scores_ = None
        self.mi_scores_ = None
        self.selected_features_ = None
        self.support_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        n_features = X_fit.shape[1]
        k = min(self.top_k, n_features)

        self.chi2_scores_, _ = chi2(X_fit, y)
        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)

        top_chi2_idx = np.argsort(self.chi2_scores_)[::-1][:k]
        top_mi_idx = np.argsort(self.mi_scores_)[::-1][:k]

        selected_idx = np.union1d(top_chi2_idx, top_mi_idx)

        self.support_ = np.zeros(n_features, dtype=bool)
        self.support_[selected_idx] = True
        self.n_features_selected_ = int(self.support_.sum())

        if self.feature_names_in_ is not None:
            self.selected_features_ = [self.feature_names_in_[i] for i in selected_idx]
        else:
            self.selected_features_ = selected_idx.tolist()

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]
        return X[:, self.support_]


# =========================================================
# Helpers
# =========================================================

def build_sampler(sampler_name="RandomOverSampler", sampling_strategy="auto", random_state=42):
    if sampler_name == "RandomOverSampler":
        return RandomOverSampler(
            sampling_strategy=sampling_strategy,
            random_state=random_state
        )
    elif sampler_name == "SMOTE":
        return SMOTE(
            sampling_strategy=sampling_strategy,
            k_neighbors=2,
            random_state=random_state
        )
    elif sampler_name is None:
        return None
    else:
        raise ValueError("sampler_name debe ser 'RandomOverSampler', 'SMOTE' o None")


def build_pipeline_with_fs_and_sampler(
    estimator,
    selector_name,
    sampler_name="RandomOverSampler",
    sampling_strategy="auto",
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
    chi2_mi_top_k=100,
    random_state=42,
):
    sampler = build_sampler(
        sampler_name=sampler_name,
        sampling_strategy=sampling_strategy,
        random_state=random_state,
    )

    steps = []

    # Selectores que NO requieren datos no negativos antes del fit
    if selector_name == "spearman_corr":
        steps.append(("selector", SpearmanCorrelationDiscard(threshold=spearman_threshold)))
        steps.append(("scaler", MinMaxScaler()))

    elif selector_name == "spearman_sulov":
        steps.append(("selector", SpearmanSULOVSelector(
            threshold=spearman_threshold,
            random_state=random_state
        )))
        steps.append(("scaler", MinMaxScaler()))

    elif selector_name == "mutual_info":
        steps.append(("selector", MIThresholdSelector(
            threshold=mi_threshold,
            random_state=random_state
        )))
        steps.append(("scaler", MinMaxScaler()))

    # Selectores que SÍ requieren no negativos
    elif selector_name == "chi2":
        steps.append(("minmax_before_chi2", MinMaxScaler()))
        steps.append(("selector", Chi2ThresholdSelector(threshold=chi2_threshold)))
        steps.append(("scaler", MinMaxScaler()))

    elif selector_name == "chi2_mi_union_topk":
        steps.append(("minmax_before_union", MinMaxScaler()))
        steps.append(("selector", Chi2MIUnionTopKSelector(
            top_k=chi2_mi_top_k,
            random_state=random_state
        )))
        steps.append(("scaler", MinMaxScaler()))

    else:
        raise ValueError(f"Selector no soportado: {selector_name}")

    # Oversampling SOLO después del preprocesado/selección y antes del modelo
    if sampler is not None:
        steps.append(("sampler", sampler))

    steps.append(("model", clone(estimator)))

    return ImbPipeline(steps)


def safe_multiclass_auc(y_true, y_proba, average="macro"):
    try:
        present_classes = np.unique(y_true)
        if len(present_classes) < 2:
            return np.nan

        y_proba_used = y_proba[:, present_classes]
        mapper = {cls: i for i, cls in enumerate(present_classes)}
        y_true_mapped = np.array([mapper[v] for v in y_true])

        return roc_auc_score(
            y_true_mapped,
            y_proba_used,
            multi_class="ovr",
            average=average
        )
    except Exception:
        return np.nan


def compute_metrics(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "AUC_macro": safe_multiclass_auc(y_true, y_proba, average="macro"),
        "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "AUC_weighted": safe_multiclass_auc(y_true, y_proba, average="weighted"),
    }


def summarize(metrics_list, suffix="CV", decimals=4):
    df = pd.DataFrame(metrics_list)
    mean = df.mean(numeric_only=True)
    std = df.std(ddof=1, numeric_only=True)

    return {
        f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
        for c in mean.index
    }


def _predict_proba_aligned(pipe, X, classes_global):
    proba = pipe.predict_proba(X)
    model_classes = pipe.named_steps["model"].classes_

    aligned = np.zeros((len(X), len(classes_global)), dtype=float)
    class_to_global_idx = {c: i for i, c in enumerate(classes_global)}

    for local_idx, cls in enumerate(model_classes):
        aligned[:, class_to_global_idx[cls]] = proba[:, local_idx]

    return aligned


# =========================================================
# Función principal: FS + Oversampling
# =========================================================

def evaluate_models_oof_and_test_with_fs_and_oversampling(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
    selectors=None,
    sampler_name: str = "RandomOverSampler",   # "RandomOverSampler", "SMOTE" o None
    sampling_strategy="auto",
    spearman_threshold: float = 0.9,
    mi_threshold: float = 0.01,
    chi2_threshold: float = 3.84,
    chi2_mi_top_k: int = 100,
):
    if selectors is None:
        selectors = [
            "spearman_corr",
            "mutual_info",
            "spearman_sulov",
            "chi2",
            "chi2_mi_union_topk",
        ]

    X = X_df.copy()
    y = y_df.iloc[:, 0].to_numpy()

    models = {
        name: model
        for name, model in models.items()
        if hasattr(model, "predict_proba")
    }

    if len(models) == 0:
        raise ValueError("Ningún modelo tiene el método predict_proba().")

    classes_global = np.unique(y)

    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    all_rows = []

    for model_name, estimator in models.items():
        for selector_name in selectors:
            print(f"Evaluating {model_name} + {selector_name} + {sampler_name}...")

            test_metrics_all = []
            cv_metrics_all = []

            for train_idx, test_idx in outer.split(X, y):
                X_train = X.iloc[train_idx].copy()
                X_test = X.iloc[test_idx].copy()
                y_train = y[train_idx]
                y_test = y[test_idx]

                _, class_counts = np.unique(y_train, return_counts=True)
                min_class_count = class_counts.min()
                effective_inner_splits = min(inner_splits, min_class_count)

                if effective_inner_splits < 2:
                    raise ValueError(
                        "No hay suficientes muestras por clase para realizar StratifiedKFold."
                    )

                inner = StratifiedKFold(
                    n_splits=effective_inner_splits,
                    shuffle=True,
                    random_state=random_state
                )

                oof_pred = np.empty(len(y_train), dtype=y_train.dtype)
                oof_proba = np.zeros((len(y_train), len(classes_global)), dtype=float)

                for tr_idx, val_idx in inner.split(X_train, y_train):
                    X_tr = X_train.iloc[tr_idx].copy()
                    X_val = X_train.iloc[val_idx].copy()
                    y_tr = y_train[tr_idx]

                    pipe = build_pipeline_with_fs_and_sampler(
                        estimator=estimator,
                        selector_name=selector_name,
                        sampler_name=sampler_name,
                        sampling_strategy=sampling_strategy,
                        spearman_threshold=spearman_threshold,
                        mi_threshold=mi_threshold,
                        chi2_threshold=chi2_threshold,
                        chi2_mi_top_k=chi2_mi_top_k,
                        random_state=random_state,
                    )

                    pipe.fit(X_tr, y_tr)

                    oof_pred[val_idx] = pipe.predict(X_val)
                    oof_proba[val_idx] = _predict_proba_aligned(pipe, X_val, classes_global)

                cv_metrics_all.append(
                    compute_metrics(
                        y_true=y_train,
                        y_pred=oof_pred,
                        y_proba=oof_proba
                    )
                )

                pipe_full = build_pipeline_with_fs_and_sampler(
                    estimator=estimator,
                    selector_name=selector_name,
                    sampler_name=sampler_name,
                    sampling_strategy=sampling_strategy,
                    spearman_threshold=spearman_threshold,
                    mi_threshold=mi_threshold,
                    chi2_threshold=chi2_threshold,
                    chi2_mi_top_k=chi2_mi_top_k,
                    random_state=random_state,
                )

                pipe_full.fit(X_train, y_train)

                test_pred = pipe_full.predict(X_test)
                test_proba = _predict_proba_aligned(pipe_full, X_test, classes_global)

                test_metrics_all.append(
                    compute_metrics(
                        y_true=y_test,
                        y_pred=test_pred,
                        y_proba=test_proba
                    )
                )

            row = {
                "Model": model_name,
                "Feature_Selection": selector_name,
                "Sampler": sampler_name if sampler_name is not None else "None",
            }

            row.update(summarize(test_metrics_all, suffix="Testing", decimals=decimals))
            row.update(summarize(cv_metrics_all, suffix="CV", decimals=decimals))

            all_rows.append(row)

    df_final_summary = pd.DataFrame(all_rows)[[
        "Model",
        "Feature_Selection",
        "Sampler",
        "F1_macro_CV",
        "F1_macro_Testing",
        "Accuracy_Testing",
        "Precision_macro_Testing",
        "Recall_macro_Testing",
        "AUC_macro_Testing",
        "Precision_weighted_Testing",
        "Recall_weighted_Testing",
        "F1_weighted_Testing",
        "AUC_weighted_Testing",
        "Accuracy_CV",
        "Precision_macro_CV",
        "Recall_macro_CV",
        "AUC_macro_CV",
        "Precision_weighted_CV",
        "Recall_weighted_CV",
        "F1_weighted_CV",
        "AUC_weighted_CV",
    ]]

    return df_final_summary

In [ ]:
for val in dominios_updrs:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43','target_HY3']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")
        if target == 'target_HY43':
                stategy_sampling={1: 150}
        else:
                stategy_sampling={2: 50}

        for val2 in ['full', 'motor']:
            if val2 == 'motor':
                dominios=dominios_updrs[val]['examen_motor']+dominios_updrs[val]['impacto_motor']
            else:
                dominios=dominios_updrs[val][val2]
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios))
            X_sub = X[dominios]
            print(f"      Data shape: {X_sub.shape}")

            for sampler in ['RandomOverSampler', 'SMOTE']:
                print(f"      Evaluating with sampler: {sampler}")
                df=evaluate_models_oof_and_test_with_fs_and_oversampling(
                                                                    X_df=X_sub, 
                                                                    y_df=y,
                                                                    models=classification_models, 
                                                                    random_state=42,
                                                                    sampler_name=sampler,
                                                                    sampling_strategy=stategy_sampling
                                                                )
                print(f"      Saving results to: {csv_output_paths['UPDRS']['FULL_SET'][target]/f'{val}_{val2}_feature_selection_{sampler}_{target}.csv'}")


Evaluating domain: X_STATS
  Target: target_HY43, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 301
      Data shape: (909, 301)
      Evaluating with sampler: RandomOverSampler
Evaluating decision_tree + spearman_corr + RandomOverSampler...
Evaluating decision_tree + mutual_info + RandomOverSampler...
Evaluating decision_tree + spearman_sulov + RandomOverSampler...
Evaluating decision_tree + chi2 + RandomOverSampler...
Evaluating decision_tree + chi2_mi_union_topk + RandomOverSampler...
Evaluating random_forest + spearman_corr + RandomOverSampler...
Evaluating random_forest + mutual_info + RandomOverSampler...
Evaluating random_forest + spearman_sulov + RandomOverSampler...
Evaluating random_forest + chi2 + RandomOverSampler...
Evaluating random_forest + chi2_mi_union_topk + RandomOverSampler...
Evaluating extra_trees + spearman_corr + RandomOverSampler...
Evaluating extra_trees + mutual_info + RandomOverSampler...
Evaluating extra_trees + spearman_sulov + Ran